In [1]:
import gymnasium
from gymnasium import spaces
import numpy as np
import pandas as pd
import yfinance as yf
from stable_baselines3 import PPO

: 

In [ ]:
# Định nghĩa môi trường giao dịch (PortfolioEnv)
class PortfolioEnv(gymnasium.Env):
    def __init__(self, data, initial_balance=1e6):
        super(PortfolioEnv, self).__init__()
        self.data = data.reset_index(drop=True)
        self.n_assets = data.shape[1]
        self.initial_balance = initial_balance
        self.current_step = 0
        
        # Hành động: phân bổ tỷ trọng cho từng tài sản (các giá trị từ 0 đến 1, tổng = 1)
        self.action_space = spaces.Box(low=0, high=1, shape=(self.n_assets,), dtype=np.float32)
        
        # Trạng thái: bao gồm giá hiện tại của tài sản và số dư hiện có
        self.observation_space = spaces.Box(low=0, high=np.inf, shape=(self.n_assets+1,), dtype=np.float32)
        
        self.balance = initial_balance

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.balance = self.initial_balance
        return self._next_observation(), {}

    def _next_observation(self):
        prices = self.data.iloc[self.current_step].values
        return np.append(prices, self.balance)

    def step(self, action):
        # Chuẩn hóa hành động để tổng các trọng số bằng 1, tránh chia cho 0
        action_sum = np.sum(action)
        if action_sum == 0:
            weights = np.ones(self.n_assets) / self.n_assets  # Phân bổ đều nếu action không hợp lệ
        else:
            weights = action / action_sum
        
        prices = self.data.iloc[self.current_step].values
        
        # Tính giá trị danh mục hiện tại
        portfolio_value = np.dot(weights, prices) * (self.balance / np.dot(np.ones(self.n_assets), prices))
        
        # Chuyển sang bước tiếp theo
        self.current_step += 1
        terminated = self.current_step >= len(self.data) - 1
        truncated = False  # Không có điều kiện dừng sớm khác
        
        next_prices = self.data.iloc[self.current_step].values
        new_portfolio_value = np.dot(weights, next_prices) * (self.balance / np.dot(np.ones(self.n_assets), prices))
        
        # Tính phần thưởng là lợi nhuận thu được
        reward = new_portfolio_value - portfolio_value
        self.balance += reward
        
        return self._next_observation(), reward, terminated, truncated, {}

    def render(self, mode='human'):
        print(f"Step: {self.current_step}, Balance: {self.balance}")

# Lấy dữ liệu lịch sử cho 4 cổ phiếu
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
data = yf.download(tickers, start="2020-01-01", end="2022-01-01")['Close']
data = data.fillna(method='ffill')  # Xử lý giá trị thiếu

# Tạo môi trường giao dịch
env = PortfolioEnv(data)

# Huấn luyện agent với thuật toán PPO
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10000)
model.save("ppo_portfolio_model")
print("Huấn luyện hoàn tất!")

# Kiểm tra mô hình đã huấn luyện
obs, _ = env.reset()
for i in range(100):
    action, _ = model.predict(obs)
    obs, reward, terminated, truncated, _ = env.step(action)
    env.render()
    if terminated or truncated:
        obs, _ = env.reset()